In [1]:
# ==========================================
# Import Libraries
# ==========================================

from pathlib import Path
import time
import joblib

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split

from sklearn.preprocessing import LabelEncoder

from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    ConfusionMatrixDisplay
)

In [3]:
# ==========================================
# Experiment Paths
# ==========================================

EXPERIMENT_DIR = Path("../experiments/E01_ModelComparison")

FIGURES_DIR = EXPERIMENT_DIR / "figures"
MODELS_DIR = EXPERIMENT_DIR / "trained_models"
CM_DIR = EXPERIMENT_DIR / "confusion_matrices"

FIGURES_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)
CM_DIR.mkdir(parents=True, exist_ok=True)

In [4]:
# ==========================================
# Load Dataset
# ==========================================

DATA_PATH = "../data/cicids2017_cleaned.csv"

chunks = pd.read_csv(
    DATA_PATH,
    chunksize=100000
)

print("Dataset loaded successfully.")

Dataset loaded successfully.


In [5]:
# ==========================================
# Balanced Sampling
# ==========================================

normal_samples = []
attack_samples = []

TARGET_PER_CLASS = 500

chunks = pd.read_csv(DATA_PATH, chunksize=100000)

for chunk in chunks:

    chunk.columns = chunk.columns.str.strip()

    normal = chunk[
        chunk["Attack Type"] == "Normal Traffic"
    ]

    attack = chunk[
        chunk["Attack Type"] != "Normal Traffic"
    ]

    if not normal.empty:
        normal_samples.append(
            normal.sample(
                min(len(normal), TARGET_PER_CLASS),
                random_state=42
            )
        )

    if not attack.empty:
        attack_samples.append(
            attack.sample(
                min(len(attack), TARGET_PER_CLASS),
                random_state=42
            )
        )

df_normal = pd.concat(
    normal_samples,
    ignore_index=True
)

df_attack = pd.concat(
    attack_samples,
    ignore_index=True
)

df_final = pd.concat(
    [df_normal, df_attack],
    ignore_index=True
)

df_final = df_final.sample(
    frac=1,
    random_state=42
).reset_index(drop=True)

print(df_final.shape)
print()
print(df_final["Attack Type"].value_counts())

(21812, 53)

Attack Type
Normal Traffic    13000
DoS                3006
Brute Force        2499
Port Scanning      1331
DDoS               1000
Bots                806
Web Attacks         170
Name: count, dtype: int64


In [14]:
# ==========================================
# Label Encoding
# ==========================================

encoder = LabelEncoder()

df_final["Attack Type Encoded"] = encoder.fit_transform(
    df_final["Attack Type"]
)

print("Class Encoding")

encoding_table = pd.DataFrame({

    "Attack Type": encoder.classes_,

    "Encoded": encoder.transform(encoder.classes_)

})

display(encoding_table)

Class Encoding


,Attack Type,Encoded
0,Bots,0
1,Brute Force,1
2,DDoS,2
3,DoS,3
4,Normal Traffic,4
5,Port Scanning,5
6,Web Attacks,6


In [15]:
# ==========================================
# Save Label Encoder
# ==========================================

import joblib

joblib.dump(

    encoder,

    EXPERIMENT_DIR / "label_encoder.pkl"

)

print("Label Encoder saved successfully.")

Label Encoder saved successfully.


In [16]:
# ==========================================
# Features and Target
# ==========================================

X = df_final.drop(
    columns=[
        "Attack Type",
        "Attack Type Encoded"
    ]
)

y = df_final["Attack Type Encoded"]

print(X.shape)
print(y.shape)

(21812, 52)
(21812,)


In [18]:
# ==========================================
# Save Feature Names
# ==========================================

joblib.dump(

    list(X.columns),

    EXPERIMENT_DIR / "feature_names.pkl"

)

print("Feature names saved successfully.")

Feature names saved successfully.


In [9]:
# ==========================================
# Train / Test Split
# ==========================================

X_train, X_test, y_train, y_test = train_test_split(

    X,

    y,

    test_size=0.30,

    random_state=42,

    stratify=y

)

print("Training Set :", X_train.shape)
print("Testing Set  :", X_test.shape)

Training Set : (15268, 52)
Testing Set  : (6544, 52)


In [10]:
# ==========================================
# Define Machine Learning Models
# ==========================================

models = {

    "Decision Tree": DecisionTreeClassifier(
        random_state=42
    ),

    "Random Forest": RandomForestClassifier(
        n_estimators=100,
        random_state=42
    ),

    "XGBoost": XGBClassifier(
        eval_metric="mlogloss",
        random_state=42
    ),

    "LightGBM": LGBMClassifier(
        random_state=42
    )

}

print(f"{len(models)} models loaded successfully.")

4 models loaded successfully.


In [11]:
# ==========================================
# Train, Evaluate and Save Models
# ==========================================

results = {}

trained_models = {}

for name, model in models.items():

    print("=" * 60)
    print(name)
    print("=" * 60)

    # -------------------------
    # Training
    # -------------------------

    start_train = time.time()

    model.fit(X_train, y_train)

    train_time = time.time() - start_train

    # -------------------------
    # Prediction
    # -------------------------

    start_pred = time.time()

    y_pred = model.predict(X_test)

    pred_time = time.time() - start_pred

    # -------------------------
    # Evaluation
    # -------------------------

    results[name] = {

        "Algorithm": name,

        "Accuracy": accuracy_score(y_test, y_pred),

        "Precision": precision_score(
            y_test,
            y_pred,
            average="weighted"
        ),

        "Recall": recall_score(
            y_test,
            y_pred,
            average="weighted"
        ),

        "F1-score": f1_score(
            y_test,
            y_pred,
            average="weighted"
        ),

        "Training Time (s)": train_time,

        "Prediction Time (s)": pred_time

    }

    trained_models[name] = model

    filename = (
        name.lower()
            .replace(" ", "_")
            + ".pkl"
    )

    joblib.dump(
        model,
        MODELS_DIR / filename
    )

print("Training completed successfully.")

Decision Tree
Random Forest
XGBoost
LightGBM
[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.013274 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 11445
[LightGBM] [Info] Number of data points in the train set: 15268, number of used features: 52
[LightGBM] [Info] Start training from score -3.298460
[LightGBM] [Info] Start training from score -2.166715
[LightGBM] [Info] Start training from score -3.082434
[LightGBM] [Info] Start training from score -1.981919
[LightGBM] [Info] Start training from score -0.517485
[LightGBM] [Info] Start training from score -2.796182
[LightGBM] [Info] Start training from score -4.854391
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain

In [12]:
# ==========================================
# Results Table
# ==========================================

results_df = pd.DataFrame(
    results.values()
)

results_df.sort_values(

    by="Accuracy",

    ascending=False,

    inplace=True

)

results_df.reset_index(

    drop=True,

    inplace=True

)

results_df

,Algorithm,Accuracy,Precision,Recall,F1-score,Training Time (s),Prediction Time (s)
0,LightGBM,0.997097,0.997106,0.997097,0.997099,7.773736,0.174116
1,XGBoost,0.997097,0.997102,0.997097,0.997098,3.319031,0.085505
2,Random Forest,0.995110,0.995117,0.995110,0.995111,11.760590,0.122876
3,Decision Tree,0.992207,0.992231,0.992207,0.992213,1.147220,0.013597


In [13]:
# ==========================================
# Save Results
# ==========================================

results_df.to_csv(

    EXPERIMENT_DIR / "comparison_results.csv",

    index=False

)

results_df.to_excel(

    EXPERIMENT_DIR / "metrics_table.xlsx",

    index=False

)

print("Results saved successfully.")

Results saved successfully.


In [19]:
# ==========================================
# Generate Comparison Figures
# ==========================================

metrics = [

    "Accuracy",

    "Precision",

    "Recall",

    "F1-score",

    "Training Time (s)",

    "Prediction Time (s)"

]

filenames = {

    "Accuracy": "accuracy_comparison.png",

    "Precision": "precision_comparison.png",

    "Recall": "recall_comparison.png",

    "F1-score": "f1_score_comparison.png",

    "Training Time (s)": "training_time_comparison.png",

    "Prediction Time (s)": "prediction_time_comparison.png"

}

for metric in metrics:

    plt.figure(figsize=(8,5))

    plt.bar(

        results_df["Algorithm"],

        results_df[metric]

    )

    plt.title(metric)

    plt.ylabel(metric)

    plt.xticks(rotation=15)

    plt.tight_layout()

    plt.savefig(

        FIGURES_DIR / filenames[metric],

        dpi=300

    )

    plt.close()

print("All comparison figures saved successfully.")

All comparison figures saved successfully.


In [20]:
# ==========================================
# Generate Confusion Matrices
# ==========================================

for name, model in trained_models.items():

    y_pred = model.predict(X_test)

    disp = ConfusionMatrixDisplay.from_predictions(

        y_test,

        y_pred,

        cmap="Blues",

        xticks_rotation=45

    )

    plt.title(name)

    filename = (

        name.lower()

        .replace(" ", "_")

        + "_cm.png"

    )

    plt.savefig(

        CM_DIR / filename,

        dpi=300,

        bbox_inches="tight"

    )

    plt.close()

print("All confusion matrices saved successfully.")

All confusion matrices saved successfully.


In [121]:
import xgboost
import sklearn
import pandas

print("xgboost:", xgboost.__version__)
print("scikit-learn:", sklearn.__version__)
print("pandas:", pandas.__version__)

xgboost: 3.1.1
scikit-learn: 1.7.2
pandas: 2.3.3
